# Emergent Introspection — Colab setup & smoke test

Runs the `safety-research/introspection-mechanisms` harness on a free Colab T4. This notebook covers **Stage 0 (environment) and Stage 0.4 (smoke test)** — the steps that need no paid GPU.

**Before you run anything:**
1. **Runtime → Change runtime type → T4 GPU.** Otherwise the model loads on CPU.
2. **Accept the Gemma license.** The smoke-test model `google/gemma-2-2b-it` is gated — accept it on its HuggingFace model page under the *same account* as your `HF_TOKEN`, or the first load will 401.
3. **Keys in Colab Secrets (🔑 sidebar):** add `HF_TOKEN` and `OPENAI_API_KEY`.

You are checking for **"no exceptions"** here — model loads, a vector is built, a generation comes out, the judge returns a label. **Not** for correct detection rates.

> 🔒 Colab disk is ephemeral, semi-public infra. Fine for the 2B smoke test (it produces none of the sensitive artifacts). **Do not mount Drive to cache vectors/activations** — regenerate, never archive.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Expect a Tesla T4 with ~15 GB. If this errors or shows no GPU, fix Runtime → Change runtime type first.

## 2. Clone the harness (public repo, no auth needed)

In [ ]:
import os
%cd /content
if not os.path.exists('introspection-mechanisms'):
    !git clone https://github.com/safety-research/introspection-mechanisms.git
else:
    print('already cloned')
%cd /content/introspection-mechanisms

## 2b. Patch a Gemma system-role bug in the harness

Gemma chat templates reject a `system` role. The harness strips system messages on its main paths but **misses its debug-capture branch**, which crashes the first trial with `TemplateError: System role not supported`. This cell applies the harness's own `filter_messages_for_model()` to the sites it forgot. Runs on the cloned file — re-run it whenever you re-clone (fresh Colab session).

In [ ]:
# Compat patch: Gemma chat templates reject a `system` role.
# The harness filters system messages on its main paths but MISSES its debug-capture
# branch, which crashes on the first trial with "System role not supported".
# This applies the harness's own filter_messages_for_model() at the sites that still
# pass a raw `messages` list. No-op for user-only calls; idempotent. Runtime-only —
# not pushed to the upstream harness (keeps their baseline code unmodified).
import re, pathlib
p = pathlib.Path("experiments/01_concept_injection.py")
src = p.read_text()
patched, n = re.subn(
    r"(apply_chat_template\(\s*\n\s*)messages,(\s*tokenize=False, add_generation_prompt=True)",
    r"\1filter_messages_for_model(messages, model.model_name),\2",
    src,
)
p.write_text(patched)
print(f"patched {n} call site(s)")   # expect 1-3

## 3. Install

`requirements.txt` pins `numpy<2.0` and Colab ships numpy 2.x, so pip will downgrade it and Colab will show a **"Restart runtime"** button. Click it, then re-run cell 2 (`%cd`) before continuing.

The heavy deps (`sae-lens`, `optuna`, `kaleido`, `networkx`) aren't needed for the smoke test or G3. If the full install is slow or fights you, comment the first line and use the slim install instead.

In [ ]:
!pip install -q -r requirements.txt
# Slim alternative (enough for smoke test + G3):
# !pip install -q torch transformers accelerate openai python-dotenv pandas "numpy<2.0" tqdm scikit-learn

## 4. Load your keys into the environment

The harness reads `HF_TOKEN` (via `transformers`) and `OPENAI_API_KEY` (the judge) from **environment variables**. This cell pulls them from Colab Secrets into `os.environ`. Never hardcode keys in the notebook — it's committed to GitHub.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# sanity check without printing the secrets:
print('HF set:', bool(os.environ.get('HF_TOKEN')), '| OpenAI set:', bool(os.environ.get('OPENAI_API_KEY')))

## 6. G3 — geometry premise check

The decisive, inference-only step — the whole hypothesis rests on it: **are harmful concept vectors more refusal-aligned than benign ones, at the injection layer?** No judge, no sweep, no abliteration: a few hundred forward passes and linear algebra. Runs on this same free T4.

A single cosine is misleading, so it sweeps three axes:
- **Extraction protocol** — last-token vs max-pool (can recover harm directions ~73° apart)
- **Layer** — all layers, read at the injection layer (~60% depth); harm/refusal coupling decouples with depth
- **Basis** — scalar `cos(v, d_refusal)` *and* projection onto the top-k refusal subspace (refusal may be a cone)

| Result at the injection layer | Meaning | Action |
|---|---|---|
| harmful **>** benign, stable across protocols | premise holds | proceed to Stage 1 |
| ordering flips between protocols | premise fragile | pre-register one protocol, proceed cautiously |
| harmful **≈** benign | premise null at injection layer | stop; use the fallback design |

The four arms (benign / negative-valence / harm-adjacent / harmful) separate **harmfulness** from **valence** and **topic** — if only the harmful arm sits high, it's harm; if valence or harm-adjacent also rise, it isn't.

> 🔒 Runs on `gemma-2-2b-it` first (geometry is recoverable even below 1B), then swap in a Gemma-3 4B to confirm. Outputs are **per-arm aggregates** — safe to keep. The concept vectors and `d_refusal` are artifacts: **in memory only — never saved to Drive, never committed.**

In [ ]:
# === G3.1 — config, concept arms, model, activation helper ===
import sys, torch, numpy as np
from tqdm.auto import tqdm
sys.path.insert(0, "src"); sys.path.insert(0, "experiments")
from transformers import AutoModelForCausalLM, AutoTokenizer

G3_MODEL       = "google/gemma-2-2b-it"  # G3 pilot model; later swap for a Gemma-3 4B to confirm
LAYER_FRACTION = 0.6      # injection depth (~60%); the 27B target injects at L37/62
N_PAIRS        = 120      # harmful/harmless instruction pairs for d_refusal
N_BASELINE     = 100      # baseline words subtracted when building concept vectors
TOPK           = 5        # refusal-subspace dimension (cone robustness)
DEVICE         = "cuda"

# Four arms (~10 concepts each) — separate harmfulness from valence and topic.
ARMS = {
    "benign":        ["bread","orchid","apple","hammer","table","cloud","river","pencil","garden","blanket"],
    "valence":       ["sadness","grief","failure","disgust","loneliness","boredom","regret","shame","misery","despair"],
    "harm_adjacent": ["police","surgery","medicine","soldier","ambulance","prison","vaccine","hospital","court","firefighter"],
    "harmful":       ["weapon","poison","assault","explosive","gun","knife","bomb","murder","torture","grenade"],
}

# Baseline words: reuse the harness's set so the geometry matches what it injects.
try:
    from vector_utils import get_baseline_words
    BASELINE_WORDS = get_baseline_words(N_BASELINE)
except Exception as e:
    print("get_baseline_words import failed, using inline fallback:", e)
    BASELINE_WORDS = ("time year people way day man thing woman life child world school state family "
                      "student group country problem hand part place case week company system program "
                      "question work government number night point home water room mother area money story "
                      "fact month lot right study book eye job word business issue side kind head house "
                      "service friend father power hour game line end member law car city community name").split()[:N_BASELINE]

tok = AutoTokenizer.from_pretrained(G3_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    G3_MODEL, torch_dtype=torch.bfloat16, device_map=DEVICE, output_hidden_states=True).eval()
N_LAYERS = model.config.num_hidden_layers
INJ = int(N_LAYERS * LAYER_FRACTION)
print(f"{G3_MODEL}: {N_LAYERS} layers · injection layer ~L{INJ} (fraction {LAYER_FRACTION})")

@torch.no_grad()
def pooled_acts(user_text):
    """Return (last_token, maxpool) residual activations, each [N_LAYERS+1, d]."""
    prompt = tok.apply_chat_template(
        [{"role": "user", "content": user_text}], tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)  # template already adds <bos>
    hs = model(**enc).hidden_states                       # tuple len N_LAYERS+1, each [1, seq, d]
    start = 1 if enc["input_ids"].shape[1] > 1 else 0     # drop BOS for max-pool
    last = torch.stack([h[0, -1] for h in hs]).float().cpu()
    mx   = torch.stack([h[0, start:].amax(0) for h in hs]).float().cpu()
    return last, mx

In [ ]:
# === G3.2 — refusal direction d_refusal + top-k refusal subspace (per layer) ===
# Arditi et al.: difference-of-means over matched harmful/harmless instructions,
# read at the last post-instruction token. Prompts imported from the harness, not hardcoded.
import refusal_prompts
HARM  = refusal_prompts.HARMFUL_PROMPTS[:N_PAIRS]
HARML = refusal_prompts.HARMLESS_PROMPTS[:N_PAIRS]

def collect_last(prompts, desc):
    return torch.stack([pooled_acts(p)[0] for p in tqdm(prompts, desc=desc)])  # [N, L+1, d]

H  = collect_last(HARM,  "harmful ")
Ml = collect_last(HARML, "harmless")

d_refusal = H.mean(0) - Ml.mean(0)                                    # [L+1, d]
d_refusal = d_refusal / (d_refusal.norm(dim=-1, keepdim=True) + 1e-8) # unit per layer

# top-k refusal subspace per layer: SVD of (harmful - mean(harmless))
U = {}
for L in range(N_LAYERS + 1):
    diff = H[:, L, :] - Ml[:, L, :].mean(0)          # [N, d]
    Vh = torch.linalg.svd(diff, full_matrices=False).Vh
    U[L] = Vh[:TOPK].T                               # [d, k]
print(f"d_refusal + top-{TOPK} subspace ready for {N_LAYERS + 1} layers")

In [ ]:
# === G3.3 — concept vectors per arm, both pooling protocols, all layers ===
# Lindsey-style, matching the harness: activation("Tell me about {word}") minus baseline mean.
_bl, _bm = [], []
for w in tqdm(BASELINE_WORDS, desc="baseline"):
    l, m = pooled_acts(f"Tell me about {w}"); _bl.append(l); _bm.append(m)
base_last, base_mx = torch.stack(_bl).mean(0), torch.stack(_bm).mean(0)   # [L+1, d] each

arm_vecs = {}   # arm -> {"last": [n, L+1, d], "maxpool": [n, L+1, d]}
for arm, words in ARMS.items():
    vl, vm = [], []
    for w in tqdm(words, desc=f"{arm:13s}"):
        l, m = pooled_acts(f"Tell me about {w}")
        vl.append(l - base_last); vm.append(m - base_mx)
    arm_vecs[arm] = {"last": torch.stack(vl), "maxpool": torch.stack(vm)}
print("concept vectors built for arms:", list(arm_vecs))

In [ ]:
# === G3.4 — cos(v_concept, d_refusal) sweep, plot, and the G3 decision ===
import matplotlib.pyplot as plt
protocols = ["last", "maxpool"]
layers = np.arange(N_LAYERS + 1)

def cos_layerwise(vecs):                                 # vecs [n, L+1, d] -> [n, L+1]
    v = vecs / (vecs.norm(dim=-1, keepdim=True) + 1e-8)
    return (v * d_refusal.unsqueeze(0)).sum(-1)

summary = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, proto in zip(axes, protocols):
    for arm in ARMS:
        c = cos_layerwise(arm_vecs[arm][proto])
        m, se = c.mean(0).numpy(), c.std(0).numpy() / np.sqrt(c.shape[0])
        ax.plot(layers, m, label=arm); ax.fill_between(layers, m - se, m + se, alpha=0.15)
        summary[(proto, arm)] = m
    ax.axvline(INJ, ls="--", c="k", alpha=0.5); ax.axhline(0, c="grey", lw=0.5)
    ax.set_title(f"protocol: {proto}"); ax.set_xlabel("layer")
axes[0].set_ylabel("cos(v_concept, d_refusal)"); axes[0].legend()
plt.suptitle(f"G3 geometry — {G3_MODEL}  (dashed = injection layer L{INJ})")
plt.tight_layout(); plt.show()

print(f"\n=== G3 decision readout at injection layer L{INJ} ===")
for proto in protocols:
    hb, bb = summary[(proto, "harmful")][INJ], summary[(proto, "benign")][INJ]
    va, ha = summary[(proto, "valence")][INJ], summary[(proto, "harm_adjacent")][INJ]
    verdict = "harmful>benign OK" if hb > bb else "harmful<=benign --"
    print(f"[{proto:8s}] harmful={hb:+.3f} benign={bb:+.3f} valence={va:+.3f} harm_adj={ha:+.3f}"
          f" | delta(harm-benign)={hb-bb:+.3f}  {verdict}")
stable = len({summary[(p, 'harmful')][INJ] > summary[(p, 'benign')][INJ] for p in protocols}) == 1
print(f"\nOrdering (harmful vs benign) stable across protocols: {stable}")

print(f"\nTop-{TOPK} refusal-subspace projection fraction at L{INJ} (last-token):")
for arm in ARMS:
    v = arm_vecs[arm]["last"][:, INJ, :]
    frac = (v @ U[INJ] @ U[INJ].T).norm(dim=-1) / (v.norm(dim=-1) + 1e-8)
    print(f"  {arm:14s} mean={frac.mean():.3f}")

print(f"""
Interpretation:
  harmful > benign at L{INJ}, stable across protocols  -> premise holds, proceed to Stage 1
  ordering flips between protocols                     -> premise fragile, pre-register one protocol
  harmful ~= benign at L{INJ}                            -> premise null at injection layer -> fallback

(Outputs above are per-arm aggregates — safe to keep. Do NOT save arm_vecs / d_refusal to Drive.)""")

In [ ]:
!python experiments/01_concept_injection.py \
    --models gemma2_2b --concepts bread hammer \
    --strength 4.0 --layer-fraction 0.6 --n-trials 2

## 6. G3 — geometry premise check *(next)*

The decisive, inference-only step: *are harmful concept vectors more refusal-aligned than benign ones at the injection layer?* This is mostly hand-written (not run through the harness) — refusal-direction extraction + concept vectors + the cosine sweep across pooling protocols and layers.

**This section will be filled in by the next push.** When it appears here after you reopen the notebook from GitHub, the sync workflow is working.